## Lesson 1: Hyperparameters and Their Role

### 1. Parameters vs Hyperparameters

* **Parameters** are learned from the data during training.
  Examples:

  * In linear regression, the coefficients (`β0, β1, …`) are parameters.
  * In a neural network, the weights of each connection are parameters.

* **Hyperparameters** are set *before* training begins. They control how the learning process unfolds, but they are not learned directly from the data.
  Examples:

  * The learning rate in gradient descent.
  * The maximum depth of a decision tree.
  * The number of neighbors in KNN.

**Key distinction:** Parameters describe the model; hyperparameters shape the *process* that finds those parameters.

### 2. Why Hyperparameters Matter

Hyperparameters directly influence the **bias-variance tradeoff**:

* **Bias**: Error from assumptions that are too strong or simplistic.

  * Example: Forcing a decision tree to `max_depth=2` creates underfitting.

* **Variance**: Error from sensitivity to small fluctuations in training data.

  * Example: Allowing a decision tree to grow fully (`max_depth=None`) leads to overfitting.

### 3. Examples of Sensitive vs Less Sensitive Hyperparameters

Not all hyperparameters are equally impactful.

* **Sensitive hyperparameters**:

  * SVM: `C`, `gamma` — drastically change decision boundaries.
  * Gradient Boosting: `learning_rate` — determines whether boosting succeeds or collapses.
  * Neural networks: `hidden_layer_sizes`, `learning_rate_init` — strongly influence learning dynamics.

* **Less sensitive hyperparameters**:

  * Random Forest: `n_estimators` — increasing usually improves performance until plateau.
  * Logistic Regression: `solver` choice — often matters only for convergence speed.

### 4. Professional Angle

In professional data science work:

* Hyperparameters determine not only accuracy, but also **training time, reproducibility, and deployment costs**.
* It’s rarely feasible to tune *every* hyperparameter exhaustively. Instead, we prioritize the impactful ones.
* A skilled practitioner knows the “key levers” for each model family.




## Lesson 2: Pipelines and Data Integrity

### 1. Why Pipelines Matter

In machine learning, the workflow usually involves multiple preprocessing steps before the model even sees the data:

* Handling missing values.
* Scaling numerical features.
* Encoding categorical features.

If these are done manually—outside the model training loop—you risk **data leakage**. That happens when information from the validation or test set “leaks” into the training process, inflating performance estimates unrealistically.

**Example of leakage**:
You scale the entire dataset with `StandardScaler` before splitting into train/test. The scaler’s mean and standard deviation are computed using *all* data—including the test set—so the model indirectly “peeks” at test data during training.

### 2. Pipelines in scikit-learn

A **Pipeline** ensures that preprocessing steps are **fitted only on the training folds** during cross-validation and then applied to validation folds. This guarantees clean evaluation.

Structure of a pipeline:

```
preprocessing → model
```

For mixed data types (numeric + categorical), we use **ColumnTransformer** inside the pipeline.

### 3. Annotated Code Example

```python
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer

# Load example dataset
X, y = load_breast_cancer(as_frame=True, return_X_y=True)

# Identify numeric and categorical columns
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

# Preprocessing pipelines
numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine into a ColumnTransformer
preprocess = ColumnTransformer([
    ("num", numeric, num_cols),
    ("cat", categorical, cat_cols)
])

# Full pipeline: preprocessing + model
pipe = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=5000))
])

# Train/test split (stratified for classification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Fit pipeline
pipe.fit(X_train, y_train)

# Evaluate
print("Test accuracy:", pipe.score(X_test, y_test))
```

### 4. Key Principles for Data Integrity

1. **Always preprocess inside a pipeline.**

   * Guarantees transformations are fit only on training data.

2. **Keep the test set untouched until the end.**

   * Use cross-validation for model selection.
   * Use test set for final evaluation only.

3. **Group-aware CV when data points aren’t independent.**

   * Example: patient-level data where multiple samples come from one patient.

### 5. Professional Practices

* Store your pipeline (`joblib.dump(pipe, "model.joblib")`) so preprocessing and model are locked together.
* This avoids the “works in notebook, fails in production” problem.
* Treat preprocessing as **part of the model**, not as a separate stage.

## Assessment for Lesson 2

### **Conceptual Questions**

Answer in your own words. If something is unclear, we’ll pause and review:

1. What’s the difference between **data leakage** and **overfitting**?
2. Why is it risky to fit a `StandardScaler` on the entire dataset before splitting into train/test?
3. In what situation would you prefer to use a **GroupKFold** rather than a StratifiedKFold?
4. Why is it considered best practice to store a pipeline object rather than just the trained model?



### **Coding Challenges**

#### Challenge 1: Leakage Detection

Below is a small script with an intentional leakage bug. Spot and correct it.

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True, as_frame=True)

# Bug: scaling before splitting
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=5000)
clf.fit(X_train, y_train)
print("Test accuracy:", clf.score(X_test, y_test))
```

**Tasks:**

* Identify why this code leaks information.
* Rewrite it using a **Pipeline** so scaling is done properly inside the cross-validation/training loop.

#### Challenge 2: Build Your Own Pipeline

You have a dataset with:

* Numeric columns: `age`, `income`.
* Categorical columns: `gender`, `region`.
  You want to predict a binary outcome with logistic regression.

**Tasks:**

* Build a ColumnTransformer that imputes numeric columns with median, scales them, imputes categorical columns with the most frequent value, and one-hot encodes them.
* Combine it with a LogisticRegression into a Pipeline.
* Split the dataset (just make up a DataFrame or use `load_breast_cancer` again for practice).
* Fit and evaluate your pipeline.
